In [1]:

!pip install chemprop

# 2. Patch astartes ให้ import ได้แม้ sklearn ยังเป็น 1.2.2
import os

file_path = "/usr/local/lib/python3.11/dist-packages/astartes/utils/user_utils.py"
with open(file_path, "r") as f:
    lines = f.readlines()

patched = []
for line in lines:
    if "from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error" in line:
        patched.append("from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error\n")
        patched.append("import numpy as np\n")
    elif "root_mean_squared_error(" in line:
        # แทนที่ function ผิด ๆ ด้วย sqrt(mean_squared_error(...))
        newline = line.replace("root_mean_squared_error(", "np.sqrt(mean_squared_error(")
        # ถ้าเปิดวงเล็บเยอะกว่า ปิดให้ครบ
        if newline.count("(") > newline.count(")"):
            newline = newline.rstrip("\n") + ")\n"
        patched.append(newline)
    else:
        patched.append(line)

with open(file_path, "w") as f:
    f.writelines(patched)




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 549.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.9/148.9 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 7.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━

In [2]:
from urllib.request import urlretrieve

urlretrieve(
    r"https://zenodo.org/records/15460715/files/chemeleon_mp.pt",
    "chemeleon_mp.pt",
)

('chemeleon_mp.pt', <http.client.HTTPMessage at 0x7f43c5e54550>)

In [3]:
import torch

from chemprop import featurizers, nn

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
agg = nn.MeanAggregation()
chemeleon_mp = torch.load("chemeleon_mp.pt", weights_only=True)
mp = nn.BondMessagePassing(**chemeleon_mp['hyper_parameters'])
mp.load_state_dict(chemeleon_mp['state_dict'])

<All keys matched successfully>

In [4]:
# Feature Extractor
import os
import random
import numpy as np
import requests
import pandas as pd
import torch

from chemprop import data, models, nn, featurizers
from lightning.pytorch import Trainer, seed_everything

#############################################
# RANDOM STATE (เพิ่มเข้มาเท่านั้น)
#############################################

random_state = 42

random.seed(random_state)
np.random.seed(random_state)
torch.manual_seed(random_state)
torch.cuda.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

seed_everything(random_state, workers=True)

#############################################
# CONFIG
#############################################

train_path = "NAI2_Train.csv"
test_path  = "NAI2_Test.csv"

checkpoint_url = "https://zenodo.org/records/15460715/files/chemeleon_mp.pt"
local_ckpt_path = "chemeleon_mp.pt"

smiles_column = "Smiles"
batch_size = 32
num_workers = 0

#############################################
# DOWNLOAD PRETRAINED
#############################################

if not os.path.exists(local_ckpt_path):
    print("⬇ Downloading pretrained weights...")
    r = requests.get(checkpoint_url)
    with open(local_ckpt_path, "wb") as f:
        f.write(r.content)
    print(" Download complete")

#############################################
# LOAD DATA (NO TARGET USED)
#############################################

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

train_data = [
    data.MoleculeDatapoint.from_smi(smi, None)
    for smi in train_df[smiles_column]
]

test_data = [
    data.MoleculeDatapoint.from_smi(smi, None)
    for smi in test_df[smiles_column]
]

#############################################
# DATASET
#############################################

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_dset = data.MoleculeDataset(train_data, featurizer)
test_dset  = data.MoleculeDataset(test_data, featurizer)

train_loader = data.build_dataloader(
    train_dset, batch_size=batch_size, num_workers=num_workers, shuffle=False
)

test_loader = data.build_dataloader(
    test_dset, batch_size=batch_size, num_workers=num_workers, shuffle=False
)

#############################################
# BUILD MODEL
#############################################

mp = nn.BondMessagePassing()
agg = nn.MeanAggregation()

ffn = nn.BinaryClassificationFFN(
    input_dim=mp.output_dim,
    hidden_dim=300,
    n_layers=2,
    dropout=0.2
)

model = models.MPNN(mp, agg, ffn, batch_norm=False)

#############################################
# LOAD PRETRAINED
#############################################

state_dict = torch.load(local_ckpt_path, map_location="cpu")
model.load_state_dict(state_dict, strict=False)

model.eval()
print(" Loaded pretrained model as Feature Extractor")

#############################################
# EXTRACT EMBEDDINGS (SAFE METHOD)
#############################################

embedding_storage = []

def hook_fn(module, input, output):
    embedding_storage.append(output.detach().cpu())

handle = model.agg.register_forward_hook(hook_fn)

trainer = Trainer(
    logger=False,
    accelerator="auto",
    devices=1
)

trainer.predict(model, train_loader)
train_emb = torch.cat(embedding_storage, dim=0)

embedding_storage.clear()

trainer.predict(model, test_loader)
test_emb = torch.cat(embedding_storage, dim=0)

handle.remove()

#############################################
# SAVE
#############################################

train_emb_df = pd.DataFrame(train_emb.numpy())
train_emb_df.to_csv("ChemPropFeatExtractor_train.csv", index=True)

test_emb_df = pd.DataFrame(test_emb.numpy())
test_emb_df.to_csv("ChemPropFeatExtractor_test.csv", index=True)

print(" Saved embedding_train.csv and embedding_test.csv")
print("Train shape:", train_emb.shape)
print("Test shape:", test_emb.shape)
print("Random state:", random_state)

INFO: Seed set to 42
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


✅ Loaded pretrained model as Feature Extractor


/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Output()

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

✅ Saved embedding_train.csv and embedding_test.csv
Train shape: torch.Size([690, 300])
Test shape: torch.Size([235, 300])
Random state: 42


In [4]:
# Finetune
import os
import random
import numpy as np
import requests
import pandas as pd
import torch

from chemprop import data, models, nn, featurizers
from lightning.pytorch import Trainer, seed_everything

#############################################
# RANDOM STATE (เพิ่มเข้ามาเท่านั้น)
#############################################

random_state = 42

random.seed(random_state)
np.random.seed(random_state)
torch.manual_seed(random_state)
torch.cuda.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

seed_everything(random_state, workers=True)

#############################################
# CONFIG
#############################################

train_path = "NAI2_Train.csv"
test_path  = "NAI2_Test.csv"

checkpoint_url = "https://zenodo.org/records/15460715/files/chemeleon_mp.pt"
local_ckpt_path = "chemeleon_mp.pt"

smiles_column = "Smiles"
batch_size = 32
num_workers = 0

#############################################
# DOWNLOAD PRETRAINED WEIGHTS
#############################################

if not os.path.exists(local_ckpt_path):
    print("⬇ Downloading pretrained weights...")
    r = requests.get(checkpoint_url)
    with open(local_ckpt_path, "wb") as f:
        f.write(r.content)
    print(" Download complete")

#############################################
# LOAD DATA
#############################################

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

train_data = [
    data.MoleculeDatapoint.from_smi(smi, [y])
    for smi, y in zip(train_df[smiles_column], train_df["Class"])
]

test_data = [
    data.MoleculeDatapoint.from_smi(smi, None)
    for smi in test_df[smiles_column]
]

#############################################
# DATASET
#############################################

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_dset = data.MoleculeDataset(train_data, featurizer)
test_dset  = data.MoleculeDataset(test_data, featurizer)

train_loader = data.build_dataloader(
    train_dset, batch_size=batch_size, num_workers=num_workers
)

test_loader = data.build_dataloader(
    test_dset, batch_size=batch_size, num_workers=num_workers, shuffle=False
)

#############################################
# BUILD MODEL (ต้อง match pretrained)
#############################################

mp = nn.BondMessagePassing()
agg = nn.MeanAggregation()

ffn = nn.BinaryClassificationFFN(
    input_dim=mp.output_dim,
    hidden_dim=300,
    n_layers=2,
    dropout=0.2
)

model = models.MPNN(mp, agg, ffn, batch_norm=False)

#############################################
# LOAD PRETRAINED
#############################################

state_dict = torch.load(local_ckpt_path, map_location="cpu")
model.load_state_dict(state_dict, strict=False)

print(" Loaded pretrained weights")

#############################################
# STAGE 1: Freeze encoder
#############################################

for param in model.message_passing.parameters():
    param.requires_grad = False

trainer = Trainer(
    logger=False,
    accelerator="auto",
    devices=1,
    max_epochs=5
)

trainer.fit(model, train_loader)

#############################################
# STAGE 2: Unfreeze full model
#############################################

for param in model.parameters():
    param.requires_grad = True

trainer = Trainer(
    logger=False,
    accelerator="auto",
    devices=1,
    max_epochs=15
)

trainer.fit(model, train_loader)

#############################################
#  EXTRACT GRAPH EMBEDDINGS (SAFE WAY)
#############################################

model.eval()

embedding_storage = []

def hook_fn(module, input, output):
    embedding_storage.append(output.detach().cpu())

handle = model.agg.register_forward_hook(hook_fn)

trainer.predict(model, train_loader)
train_emb = torch.cat(embedding_storage, dim=0)

embedding_storage.clear()

trainer.predict(model, test_loader)
test_emb = torch.cat(embedding_storage, dim=0)

handle.remove()

#############################################
# SAVE
#############################################

assert train_emb.shape[0] == len(train_df)
assert test_emb.shape[0] == len(test_df)

train_emb_df = pd.DataFrame(train_emb.numpy())
train_emb_df["label"] = train_df["Class"].values
train_emb_df.to_csv("ChemPropFineTune_train.csv", index=True)

test_emb_df = pd.DataFrame(test_emb.numpy())
test_emb_df.to_csv("ChemPropFineTune_test.csv", index=True)

print(" Saved REAL embedding_train.csv and embedding_test.csv")
print("Train shape:", train_emb.shape)
print("Test shape:", test_emb.shape)
print("Random state:", random_state)

INFO: Seed set to 42
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


✅ Loaded pretrained weights


/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │  180 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 180 K                                                                                            
Non-trainable params: 227 K                                                                                        
Total params: 408 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 26                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Output()

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:485: Your `predict_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

✅ Saved REAL embedding_train.csv and embedding_test.csv
Train shape: torch.Size([690, 300])
Test shape: torch.Size([173, 300])
Random state: 42


In [ ]:
trainer.fit(mpnn, train_loader, val_loader)


In [ ]:
results = trainer.test(dataloaders=test_loader)
